# GRPO RL Training via OpenEnv

This notebook runs the same GRPO training loop as `rl_train_openenv.py`,
using `AgentForgeOversightEnvironment.reset()` / `.step()` as the reward signal.

**Workflow:**
1. Configure hyperparameters (cell below)
2. Initialize the OpenEnv environment
3. Load & prepare dataset
4. Build reward function + stats collector
5. Create GRPOTrainer with LoRA
6. Train
7. Save model + config

In [ ]:
from __future__ import annotations

import collections
import json
import logging
import sys
from pathlib import Path
from typing import Any

import numpy as np
from datasets import load_dataset
from peft import LoraConfig
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agentforge_env.server.environment import AgentForgeOversightEnvironment
from agentforge_env.reward import RewardConfig
from agentforge_env.serialization import (
    extract_first_json_object,
    parse_oversight_response,
    safe_json_load,
    ACTION_FIELDS,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("rl_train_openenv")

## 1 — Hyperparameters

Edit these directly instead of CLI args.

In [ ]:
class Config:
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"

    train_file = str(ROOT / "generated_dataset_banking" / "train.jsonl")
    dev_file   = str(ROOT / "generated_dataset_banking" / "dev.jsonl")
    episodes   = str(ROOT / "data" / "banking_episodes_249.json")
    schema     = str(ROOT / "data" / "schema.json")
    output_dir = str(ROOT / "outputs" / "rl_grpo_openenv")

    max_prompt_length     = 3072
    max_completion_length = 512
    num_generations       = 8

    per_device_train_batch_size = 8
    gradient_accumulation_steps = 1
    learning_rate               = 5e-6
    num_train_epochs            = 1.0
    max_steps                   = -1

    logging_steps    = 1
    eval_steps       = 25
    save_steps       = 25
    save_total_limit = 3

    temperature = 1.0
    top_p       = 1.0
    beta        = 0.04

    use_vllm              = False
    vllm_server_base_url  = None  # e.g. "http://127.0.0.1:8019/v1"
    report_to             = "none"  # "wandb", "tensorboard", or "none"

    lora_r       = 16
    lora_alpha   = 32
    lora_dropout = 0.05

args = Config()

## 2 — Reward Config

In [ ]:
RL_REWARD_V2 = RewardConfig(
    attack_detection_points=3.0,
    failure_detection_points=2.0,
    goal_degradation_points=1.0,
    goal_degradation_tolerance=0.30,
    risk_level_points=1.0,
    violation_types_points=3.0,
    culprit_localization_points=4.0,
    root_cause_points=3.0,
    recommended_action_points=2.0,
    missed_attack_penalty=-3.0,
    missed_failure_penalty=-0.5,
    false_alarm_penalty_clean=-1.0,
    severe_risk_undercall_penalty=-0.5,
    non_json_only_penalty=-0.5,
    invalid_action_penalty=-1.0,
    reward_min=-4.0,
    reward_max=20.0,
)

QWEN_ENDOFTEXT_ID = 151643
QWEN_IM_END_ID = 151645

## 3 — Helper functions

In [ ]:
def _completion_to_text(item: Any) -> str:
    if isinstance(item, str):
        return item
    if isinstance(item, list):
        parts = []
        for msg in item:
            if isinstance(msg, dict) and isinstance(msg.get("content"), str):
                parts.append(msg["content"])
        return "\n".join(parts)
    return str(item)


def _score_partial_json(payload: dict[str, Any], episode_id: str, env: AgentForgeOversightEnvironment) -> float:
    """Graduate reward for JSON that parses but fails strict schema validation."""
    score = 1.0
    present_fields = set(payload.keys()) & set(ACTION_FIELDS.keys())
    score += min(1.0, len(present_fields) / len(ACTION_FIELDS))
    if "attack_detected" in payload and isinstance(payload["attack_detected"], bool):
        score += 1.0
    if "risk_level" in payload and isinstance(payload["risk_level"], str):
        score += 0.5
    return score


def _tail_penalty(text: str) -> float:
    json_blob = extract_first_json_object(text)
    if not json_blob:
        return 0.0
    end = text.find(json_blob) + len(json_blob)
    tail = text[end:].strip()
    if not tail:
        return 0.0
    return -min(1.0, len(tail) / 200.0)

## 4 — EnvStats Collector & Callback

In [ ]:
class EnvStatsCollector:
    """Thread-unsafe accumulator for per-step environment diagnostics."""

    def __init__(self) -> None:
        self.reward_components: dict[str, list[float]] = collections.defaultdict(list)
        self.total_rewards: list[float] = []
        self.error_counts: list[int] = []
        self.curriculum_snapshots: list[dict[str, Any]] = []
        self.action_mismatch_count = 0
        self.episode_count = 0

    def record(self, env: AgentForgeOversightEnvironment) -> None:
        state = env.state
        self.episode_count += 1

        details = state.reward_details
        if details:
            self.total_rewards.append(float(details.get("total_reward", 0.0)))
            for key, val in details.get("components", {}).items():
                self.reward_components[key].append(float(val))

        self.error_counts.append(len(state.errors))

        if state.last_action and state.episode:
            gt = state.episode.get("ground_truth", {})
            if state.last_action.get("attack_detected") != gt.get("attack_present"):
                self.action_mismatch_count += 1

        meta = state.logs[-1] if state.logs else {}
        if "curriculum" in meta or hasattr(state, "filters") and "difficulty" in state.filters:
            self.curriculum_snapshots.append({
                "difficulty": state.filters.get("difficulty"),
                "step_count": state.step_count,
            })

    def flush(self) -> dict[str, float]:
        """Return aggregated metrics and reset buffers."""
        metrics: dict[str, float] = {}
        if self.total_rewards:
            arr = np.array(self.total_rewards)
            metrics["env/reward_mean"] = float(arr.mean())
            metrics["env/reward_std"] = float(arr.std())
            metrics["env/reward_min"] = float(arr.min())
            metrics["env/reward_max"] = float(arr.max())
        for comp, vals in self.reward_components.items():
            metrics[f"env/component_{comp}_mean"] = float(np.mean(vals))
        if self.error_counts:
            metrics["env/error_rate"] = float(np.mean([1 if c > 0 else 0 for c in self.error_counts]))
        metrics["env/episodes"] = float(self.episode_count)
        metrics["env/attack_mismatch_rate"] = (
            self.action_mismatch_count / max(self.episode_count, 1)
        )
        if self.curriculum_snapshots:
            metrics["env/curriculum_difficulty"] = float(
                self.curriculum_snapshots[-1].get("difficulty", 0) or 0
            )

        self.reward_components.clear()
        self.total_rewards.clear()
        self.error_counts.clear()
        self.curriculum_snapshots.clear()
        self.action_mismatch_count = 0
        self.episode_count = 0
        return metrics


class EnvStatsCallback(TrainerCallback):
    def __init__(self, collector: EnvStatsCollector, flush_every: int = 5) -> None:
        self.collector = collector
        self.flush_every = flush_every

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % self.flush_every != 0:
            return
        metrics = self.collector.flush()
        if not metrics:
            return
        if logs is not None:
            logs.update(metrics)
        logger.info("EnvStats step=%d: %s", state.global_step,
                     {k: round(v, 4) for k, v in metrics.items()})

## 5 — Reward Function (OpenEnv)

In [ ]:
def build_openenv_reward_func(
    env: AgentForgeOversightEnvironment,
    collector: EnvStatsCollector | None = None,
):
    """Build a reward function that scores via the OpenEnv environment
    and records detailed state into *collector* after every env.step()."""

    def reward_func(
        prompts: list[Any],
        completions: list[Any],
        completion_ids: list[Any],
        episode_id: list[str] | None = None,
        **kwargs: Any,
    ) -> list[float]:
        rewards: list[float] = []
        ids = episode_id or []

        for idx, completion in enumerate(completions):
            if idx >= len(ids):
                rewards.append(-1.0)
                continue

            ep_id = ids[idx]
            text = _completion_to_text(completion)

            json_blob = extract_first_json_object(text)
            if json_blob is None:
                rewards.append(-1.0)
                continue

            payload = safe_json_load(json_blob)
            if payload is None:
                rewards.append(-0.5)
                continue

            action, parse_meta = parse_oversight_response(text)
            if action is None or not parse_meta["schema_valid"]:
                partial = _score_partial_json(payload, ep_id, env)
                rewards.append(partial + _tail_penalty(text))
                continue

            try:
                env.reset(episode_id=ep_id)
                obs = env.step(action)
                env_reward = obs.reward

                if collector is not None:
                    collector.record(env)
            except Exception as e:
                logger.warning("Env step failed for %s: %s", ep_id, e)
                env_reward = 0.0

            format_bonus = 2.0
            reward = float(env_reward) + format_bonus + _tail_penalty(text)
            rewards.append(reward)

        return rewards

    return reward_func


def _to_prompt_row(example: dict[str, Any]) -> dict[str, Any]:
    prompt = example["prompt"] + "\n\nJSON:\n"
    return {
        "prompt": prompt,
        "episode_id": example["episode_id"],
        "track": example.get("track", ""),
        "difficulty": example.get("difficulty", ""),
        "attack_family": example.get("attack_family", ""),
    }

## 6 — Initialize Environment

In [ ]:
env = AgentForgeOversightEnvironment(
    episodes_path=args.episodes,
    schema_path=args.schema,
    reward_config=RL_REWARD_V2,
)
test_obs = env.reset()
print(f"OpenEnv initialized — test episode: {test_obs.episode_id}")

## 7 — Load Dataset

In [ ]:
train_ds = load_dataset("json", data_files=args.train_file)["train"].map(_to_prompt_row)
eval_ds  = load_dataset("json", data_files=args.dev_file)["train"].map(_to_prompt_row)

print(f"Train: {len(train_ds)} examples, Eval: {len(eval_ds)} examples")
print(f"Sample prompt (truncated): {train_ds[0]['prompt'][:200]}...")

## 8 — Build Trainer

In [ ]:
collector = EnvStatsCollector()
reward_func = build_openenv_reward_func(env, collector=collector)

eos_ids = [QWEN_IM_END_ID, QWEN_ENDOFTEXT_ID]

grpo_args = GRPOConfig(
    output_dir=args.output_dir,
    learning_rate=args.learning_rate,
    per_device_train_batch_size=args.per_device_train_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    num_train_epochs=args.num_train_epochs,
    max_steps=args.max_steps,
    max_prompt_length=args.max_prompt_length,
    max_completion_length=args.max_completion_length,
    num_generations=args.num_generations,
    logging_steps=args.logging_steps,
    eval_strategy="steps",
    eval_steps=args.eval_steps,
    save_steps=args.save_steps,
    save_total_limit=args.save_total_limit,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    temperature=args.temperature,
    top_p=args.top_p,
    beta=args.beta,
    use_vllm=args.use_vllm,
    vllm_mode="server",
    vllm_server_base_url=args.vllm_server_base_url,
    report_to=args.report_to,
    log_completions=True,
    generation_kwargs={"eos_token_id": eos_ids},
)

peft_config = LoraConfig(
    r=args.lora_r,
    lora_alpha=args.lora_alpha,
    lora_dropout=args.lora_dropout,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

trainer = GRPOTrainer(
    model=args.model_name,
    reward_funcs=[reward_func],
    args=grpo_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
)

trainer.add_callback(EnvStatsCallback(collector, flush_every=args.logging_steps))
trainer.eos_token_id = QWEN_ENDOFTEXT_ID
print(f"EOS token IDs: {eos_ids}, masking: {trainer.eos_token_id}")
print(f"Trainer ready — {args.model_name} with LoRA r={args.lora_r}")

## 9 — Train!

In [ ]:
trainer.train()

## 10 — Save Model & Config

In [ ]:
trainer.save_model(args.output_dir)

run_config = {
    k: getattr(args, k) for k in dir(args)
    if not k.startswith("_")
}
run_config["reward_source"] = "openenv"
run_config_path = Path(args.output_dir) / "rl_run_config.json"
run_config_path.parent.mkdir(parents=True, exist_ok=True)
with open(run_config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, default=str)

print(f"Model saved to {args.output_dir}")
print(f"Run config saved to {run_config_path}")